In [ ]:
import polars as pl
import polars.selectors as cs

from aare_train.params import read_params
from aare_train.paths import DATA_FOLDER

# Accuracy difference between measurement eval and inference

Since we are using measurement data as training, validation and test data, the model not only takes the perfect accuracy of the air temperature for granted, but also our evaluation metrics are calculated in the most optimal situation.
In reality, the air temperature future covariate are weather forecasts from MeteoTest. They don't state how accurate their forecasts are, but since we historize everything, we can check ourselves. \
Additionally, we would like to correct our test set eval. The test set eval is done to get an estimate of how well we can expect the model to perform in the real world after deployment.
However, since we have a misalignment of test data and real data, we must assume that our calculated 'expected' accuracy is very optimistic and higher than it will actually perform.
How much worse it will actually perform depends on how good the forecasts of MeteoTest are (= how big the misalignment is).
To correct for this, we can take forecasts the model prototype has made so far and simulate a test set eval for the same model in same time period.
Then calculate how much worse forecasts are and try to extrapolate that for other models and into summer.
We can assume that the difference gets bigger as we transition from winter to summer and the further we forecast, since weather forecasts most likely also struggle more during those times than for example with short-term winter forecasts.
This means inaccuracies will compound and our forecasts 3-4 days into the future could be completely unusable (that also why the MVP only does 24h max).

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
params = read_params()
tz = params["general"]["timezone"]

In [ ]:
df_inf = pl.read_csv(DATA_FOLDER / "backups/forecast.csv")
df_inf

In [ ]:
df_inf = df_inf.with_columns(cs.string().str.to_datetime(time_zone=tz))
df_inf

In [ ]:
df_eval = pl.read_parquet(DATA_FOLDER / "metrics/raw/LR-dev-quasi-prod-test.parquet")
df_eval

In [ ]:
pl.select(pl.max_horizontal(df_inf.select(pl.min("run_ts")), df_eval.select(pl.min("run_ts"))))

In [ ]:
common_start = max(df_inf.select(pl.min("run_ts")).item(), df_eval.select(pl.min("run_ts")).item())
common_start

In [ ]:
common_end = min(df_inf.select(pl.max("run_ts")).item(), df_eval.select(pl.max("run_ts")).item())
common_end

In [ ]:
def assimilate(df: pl.DataFrame):
    df = df.lazy()
    run_ts = pl.col("run_ts")
    df = df.filter(run_ts >= common_start, run_ts <= common_end)
    # only keep the latest run for each hour, since inference does 4 per hour. eval only has 1 per hour anyway.
    df = df.filter(run_ts.dt.minute() >= 40)
    # get the hour of the run_ts (should always be the first predicted timestamp of the run - 1h).
    # Note that I tried doing this with 'dt.replace', but that creates a new timestamp in the context of the timezone without
    # context of the original point in global time, so it thinks it's an ambiguous timestamp if we land on a
    # daylight savings hour. So either go UTC, replace and go back to CET, or (what I thought of earlier) subtract the hours, min and sec.
    # df = df.with_columns(run_hour=pl.col("run_ts").dt.replace(minute=0, second=0, microsecond=0))
    df = df.with_columns(
        run_hour=run_ts
        - pl.duration(minutes=run_ts.dt.minute(), seconds=run_ts.dt.second(), microseconds=run_ts.dt.microsecond())
    )

    df = df.collect()

    return df

In [ ]:
assimilate(df_inf)

In [ ]:
assimilate(df_eval)